# 04 – Feature Engineering: Variables Demográficas

**Proyecto:** Predicción de Subempleo por Insuficiencia de Horas — EPEN 2024  
**Etapa:** Feature Engineering – Paso 1 de 4  
**Dataset de entrada:** `data/processed/epen_variable_filtered.csv`  
**Dataset de salida:** `data/feature_engineering/epen_fe_demographic.csv`

## Objetivo

Crear variables derivadas de características demográficas del trabajador: **sexo**, **edad**, **parentesco con el jefe del hogar**, **discapacidad** y **etnicidad**.

### Consideraciones metodológicas
- El *feature engineering* conceptual se realiza **antes** del train/test split porque se basa en reglas fijas, no en información aprendida del conjunto de datos.
- Las variables sensibles (etnicidad, discapacidad) se incluyen para capturar brechas estructurales relacionadas con el subempleo.
- El target **no** se usa para construir features.

### Restricciones
- No se usa: `P209H`, `C333`, `C334`, `fa_son24`
- No se realiza train/test split en este notebook
- No se realiza balanceo ni selección estadística de variables

---
## 1. Cargar Librerías

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


---
## 2. Cargar Dataset

In [2]:
INPUT_PATH = Path('../data/processed/epen_variable_filtered.csv')

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró el archivo de entrada: {INPUT_PATH}\n'
        'Ejecuta primero: 03_preprocessing/03_variable_filtering.ipynb'
    )

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Dataset cargado : {INPUT_PATH.name}')
print(f'Dimensiones     : {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'\nColumnas disponibles ({df.shape[1]}):')
print(df.columns.tolist())

Dataset cargado : epen_variable_filtered.csv
Dimensiones     : 24,054 filas x 46 columnas

Columnas disponibles (46):
['C207', 'C208', 'C203', 'C366', 'C366_1', 'C366_2', 'C310', 'C311', 'C312', 'C313', 'C317', 'C317A', 'C335', 'C318_T', 'C328_T', 'whoraT', 'C331', 'INGTOT', 'INGTOTP', 'ingtrabw', 'I339_1', 'I342', 'I345_1', 'I348', 'INGTOT_log', 'INGTOTP_log', 'INGTRABW_log', 'SEGURO1', 'C361_1', 'C361_5', 'C364_1', 'C364_2', 'C375_1', 'C375_2', 'C375_3', 'C375_4', 'C375_5', 'C375_6', 'C376', 'C377', 'whoraT_missing_flag', 'C318_T_missing_flag', 'whoraT_outlier_flag', 'INGTOT_outlier_flag', 'INGTRABW_outlier_flag', 'target_subempleo_horas']


---
## 3. Validación Inicial

In [3]:
# Verificar target
assert 'target_subempleo_horas' in df.columns, \
    "ERROR: 'target_subempleo_horas' no encontrado en el dataset de entrada."
assert df['target_subempleo_horas'].isnull().sum() == 0, \
    'ERROR: target_subempleo_horas contiene valores nulos.'
print('target_subempleo_horas presente y sin nulos: OK')

# Verificar ausencia de variables de leakage
for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df.columns, f'ERROR: variable de leakage {lv} encontrada.'
print('Variables de leakage ausentes: OK')

# Distribución del target
counts = df['target_subempleo_horas'].value_counts()
pct    = df['target_subempleo_horas'].value_counts(normalize=True) * 100
print('\nDistribución del target:')
display(pd.DataFrame({'conteo': counts, 'porcentaje (%)': pct.round(2)}))

target_subempleo_horas presente y sin nulos: OK
Variables de leakage ausentes: OK

Distribución del target:


,conteo,porcentaje (%)
target_subempleo_horas,,
0,18064,75.1000
1,5990,24.9000


---
## 4. Crear Copia de Trabajo

In [4]:
df_fe = df.copy()
n_original = df_fe.shape[0]
features_created = []

print(f'Copia creada: {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')

Copia creada: 24,054 filas x 46 columnas


---
## 5. Variables de Sexo

**Variable base:** `C207`

| Código | Significado |
|:------:|:------------|
| 1 | Hombre |
| 2 | Mujer |

**Features creados:** `sexo_mujer`, `sexo_hombre`

In [5]:
if 'C207' in df_fe.columns:
    df_fe['sexo_mujer']  = (df_fe['C207'] == 2).astype(int)
    df_fe['sexo_hombre'] = (df_fe['C207'] == 1).astype(int)
    features_created += ['sexo_mujer', 'sexo_hombre']

    print(f"sexo_mujer  : {df_fe['sexo_mujer'].sum():>7,}  ({df_fe['sexo_mujer'].mean()*100:.1f}%)")
    print(f"sexo_hombre : {df_fe['sexo_hombre'].sum():>7,}  ({df_fe['sexo_hombre'].mean()*100:.1f}%)")

    # Tasa de subempleo por sexo
    print('\nTasa de subempleo por sexo:')
    display(df_fe.groupby('sexo_mujer')['target_subempleo_horas'].mean().rename('tasa_subempleo').to_frame())
else:
    print('ADVERTENCIA: C207 no encontrado. Variables sexo_mujer y sexo_hombre no creadas.')

sexo_mujer  :  11,310  (47.0%)
sexo_hombre :  12,744  (53.0%)

Tasa de subempleo por sexo:


,tasa_subempleo
sexo_mujer,
0,0.2781
1,0.2163


---
## 6. Variables de Edad

**Variable base:** `C208` (edad en años cumplidos)

**Features creados:** `edad`, `grupo_edad`, `joven` (14–29 años), `adulto_mayor` (≥65 años)

In [6]:
if 'C208' in df_fe.columns:
    df_fe['edad'] = pd.to_numeric(df_fe['C208'], errors='coerce')

    bins   = [0, 24, 34, 44, 54, 64, 200]
    labels = ['14_24', '25_34', '35_44', '45_54', '55_64', '65_mas']
    df_fe['grupo_edad'] = pd.cut(df_fe['edad'], bins=bins, labels=labels, right=True)

    df_fe['joven']        = ((df_fe['edad'] >= 14) & (df_fe['edad'] <= 29)).astype(int)
    df_fe['adulto_mayor'] = (df_fe['edad'] >= 65).astype(int)

    features_created += ['edad', 'grupo_edad', 'joven', 'adulto_mayor']

    print('Distribución por grupo de edad:')
    edad_dist = df_fe['grupo_edad'].value_counts().sort_index()
    print(edad_dist)
    print(f"\njoven        : {df_fe['joven'].sum():,}")
    print(f"adulto_mayor : {df_fe['adulto_mayor'].sum():,}")

    print('\nTasa de subempleo por grupo de edad:')
    display(df_fe.groupby('grupo_edad', observed=True)['target_subempleo_horas'].mean().rename('tasa_subempleo').to_frame())
else:
    print('ADVERTENCIA: C208 no encontrado. Variables de edad no creadas.')

Distribución por grupo de edad:
grupo_edad
14_24     3193
25_34     5154
35_44     5506
45_54     4871
55_64     3486
65_mas    1844
Name: count, dtype: int64

joven        : 5,718
adulto_mayor : 1,844

Tasa de subempleo por grupo de edad:


,tasa_subempleo
grupo_edad,
14_24,0.2205
25_34,0.2416
35_44,0.2684
45_54,0.2677
55_64,0.2493
65_mas,0.2115


---
## 7. Variables de Parentesco con el Jefe del Hogar

**Variable base:** `C203`

| Código | Significado |
|:------:|:------------|
| 1 | Jefe/a del hogar |
| 2 | Cónyuge / conviviente |
| 3 | Hijo/a |

**Features creados:** `jefe_hogar`, `conyuge`, `hijo_hogar`

In [7]:
if 'C203' in df_fe.columns:
    df_fe['jefe_hogar'] = (df_fe['C203'] == 1).astype(int)
    df_fe['conyuge']    = (df_fe['C203'] == 2).astype(int)
    df_fe['hijo_hogar'] = (df_fe['C203'] == 3).astype(int)
    features_created   += ['jefe_hogar', 'conyuge', 'hijo_hogar']

    print(f"jefe_hogar : {df_fe['jefe_hogar'].sum():>7,}  ({df_fe['jefe_hogar'].mean()*100:.1f}%)")
    print(f"conyuge    : {df_fe['conyuge'].sum():>7,}  ({df_fe['conyuge'].mean()*100:.1f}%)")
    print(f"hijo_hogar : {df_fe['hijo_hogar'].sum():>7,}  ({df_fe['hijo_hogar'].mean()*100:.1f}%)")
else:
    print('ADVERTENCIA: C203 no encontrado. Variables de parentesco no creadas.')

jefe_hogar :  11,088  (46.1%)
conyuge    :   4,683  (19.5%)
hijo_hogar :   6,321  (26.3%)


---
## 8. Variables de Discapacidad

**Variables base:** `C375_1` a `C375_6` (limitaciones permanentes)  
Codificación: **1 = Sí** tiene la limitación, **2 = No** tiene la limitación

| Variable | Limitación |
|:--------:|:-----------|
| C375_1 | Para ver |
| C375_2 | Para oír |
| C375_3 | Para hablar o comunicarse |
| C375_4 | Para moverse o caminar |
| C375_5 | Para entender o aprender |
| C375_6 | Para relacionarse con los demás |

**Features creados:** `tiene_discapacidad`, `cantidad_limitaciones`

In [8]:
discapacidad_cols    = [f'C375_{i}' for i in range(1, 7)]
discapacidad_present = [c for c in discapacidad_cols if c in df_fe.columns]

if discapacidad_present:
    disc_binary = (df_fe[discapacidad_present] == 1).astype(int)

    df_fe['tiene_discapacidad']    = (disc_binary.max(axis=1) == 1).astype(int)
    df_fe['cantidad_limitaciones'] = disc_binary.sum(axis=1)
    features_created += ['tiene_discapacidad', 'cantidad_limitaciones']

    print(f'Columnas de discapacidad encontradas: {discapacidad_present}')
    print(f"\ntiene_discapacidad   : {df_fe['tiene_discapacidad'].sum():,}  ({df_fe['tiene_discapacidad'].mean()*100:.2f}%)")
    print(f"\ncantidad_limitaciones:")
    print(df_fe['cantidad_limitaciones'].value_counts().sort_index())
else:
    print(f'ADVERTENCIA: Ninguna columna de discapacidad encontrada. Buscadas: {discapacidad_cols}')

Columnas de discapacidad encontradas: ['C375_1', 'C375_2', 'C375_3', 'C375_4', 'C375_5', 'C375_6']

tiene_discapacidad   : 193  (0.80%)

cantidad_limitaciones:
cantidad_limitaciones
0    23861
1      176
2       14
3        1
5        1
6        1
Name: count, dtype: int64


---
## 9. Variables de Etnicidad

**Variables base:**
- `C376`: lengua o idioma aprendido en la niñez
- `C377`: autoidentificación étnica

| C376 | Lengua |
|:----:|:-------|
| 1–9 | Lenguas originarias (quechua, aymara, asháninka, etc.) |
| 10 | Castellano |
| 11+ | Lengua extranjera / señas |

| C377 | Autoidentificación |
|:----:|:-------------------|
| 1–4 | Pueblos indígenas (quechua, aymara, amazónico, otro) |
| 5 | Afroperuano/a |
| 6 | Blanco/a |
| 7 | Mestizo/a |
| 8 | Otro |

**Features creados:** `lengua_materna_indigena`, `autoidentificacion_indigena`, `autoidentificacion_afro`, `autoidentificacion_mestizo`

In [9]:
if 'C376' in df_fe.columns:
    df_fe['lengua_materna_indigena'] = df_fe['C376'].between(1, 9).astype(int)
    features_created.append('lengua_materna_indigena')
    pct = df_fe['lengua_materna_indigena'].mean() * 100
    print(f"lengua_materna_indigena : {df_fe['lengua_materna_indigena'].sum():,}  ({pct:.2f}%)")
else:
    print('ADVERTENCIA: C376 no encontrado.')

if 'C377' in df_fe.columns:
    df_fe['autoidentificacion_indigena'] = df_fe['C377'].isin([1, 2, 3, 4]).astype(int)
    df_fe['autoidentificacion_afro']     = (df_fe['C377'] == 5).astype(int)
    df_fe['autoidentificacion_mestizo']  = (df_fe['C377'] == 7).astype(int)
    features_created += ['autoidentificacion_indigena', 'autoidentificacion_afro', 'autoidentificacion_mestizo']

    print(f"autoidentificacion_indigena : {df_fe['autoidentificacion_indigena'].sum():,}  ({df_fe['autoidentificacion_indigena'].mean()*100:.2f}%)")
    print(f"autoidentificacion_afro     : {df_fe['autoidentificacion_afro'].sum():,}  ({df_fe['autoidentificacion_afro'].mean()*100:.2f}%)")
    print(f"autoidentificacion_mestizo  : {df_fe['autoidentificacion_mestizo'].sum():,}  ({df_fe['autoidentificacion_mestizo'].mean()*100:.2f}%)")
else:
    print('ADVERTENCIA: C377 no encontrado.')

lengua_materna_indigena : 2,132  (8.86%)
autoidentificacion_indigena : 3,195  (13.28%)
autoidentificacion_afro     : 818  (3.40%)
autoidentificacion_mestizo  : 17,964  (74.68%)


---
## 10. Validaciones Finales

In [10]:
# Verificar que no cambió el número de filas
assert df_fe.shape[0] == n_original, \
    f'ERROR: el número de filas cambió. Original: {n_original}, actual: {df_fe.shape[0]}'
print(f'Número de filas sin cambios : OK ({n_original:,})')

# Verificar presencia del target
assert 'target_subempleo_horas' in df_fe.columns, 'ERROR: target_subempleo_horas no presente.'
print('target_subempleo_horas presente: OK')

# Verificar ausencia de leakage
for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df_fe.columns, f'ERROR: variable de leakage {lv} presente.'
print('Variables de leakage ausentes: OK')

# Resumen de features creados
print(f'\nFeatures demograficos creados ({len(features_created)}):')
for feat in features_created:
    nulos = df_fe[feat].isnull().sum()
    n_uniq = df_fe[feat].nunique()
    print(f'  {feat:<38}: {nulos:>6} nulos  ({nulos/len(df_fe)*100:.2f}%)  | {n_uniq} valores únicos')

print(f'\nDimensiones finales  : {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')
print(f'Columnas nuevas      : {df_fe.shape[1] - df.shape[1]}')

Número de filas sin cambios : OK (24,054)
target_subempleo_horas presente: OK
Variables de leakage ausentes: OK

Features demograficos creados (15):
  sexo_mujer                            :      0 nulos  (0.00%)  | 2 valores únicos
  sexo_hombre                           :      0 nulos  (0.00%)  | 2 valores únicos
  edad                                  :      0 nulos  (0.00%)  | 78 valores únicos
  grupo_edad                            :      0 nulos  (0.00%)  | 6 valores únicos
  joven                                 :      0 nulos  (0.00%)  | 2 valores únicos
  adulto_mayor                          :      0 nulos  (0.00%)  | 2 valores únicos
  jefe_hogar                            :      0 nulos  (0.00%)  | 2 valores únicos
  conyuge                               :      0 nulos  (0.00%)  | 2 valores únicos
  hijo_hogar                            :      0 nulos  (0.00%)  | 2 valores únicos
  tiene_discapacidad                    :      0 nulos  (0.00%)  | 2 valores únicos
  cantidad

---
## 11. Guardar Resultados

In [11]:
OUTPUT_DIR = Path('../data/feature_engineering')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset con features demográficas
out_main = OUTPUT_DIR / 'epen_fe_demographic.csv'
df_fe.to_csv(out_main, index=False)
print(f'Dataset guardado : {out_main}  ({df_fe.shape[0]:,} x {df_fe.shape[1]})')

# Reporte de features creados
report = pd.DataFrame({
    'feature'    : features_created,
    'tipo_dato'  : [str(df_fe[f].dtype) for f in features_created],
    'nulos'      : [df_fe[f].isnull().sum() for f in features_created],
    'n_unicos'   : [df_fe[f].nunique() for f in features_created],
    'pct_nulos'  : [round(df_fe[f].isnull().mean() * 100, 4) for f in features_created],
})
out_report = OUTPUT_DIR / 'demographic_features_created.csv'
report.to_csv(out_report, index=False)
print(f'Reporte guardado : {out_report}  ({len(report)} features)')

print('\nTodos los archivos guardados correctamente.')
print('Siguiente paso -> 02_education_features.ipynb')

Dataset guardado : ..\data\feature_engineering\epen_fe_demographic.csv  (24,054 x 61)
Reporte guardado : ..\data\feature_engineering\demographic_features_created.csv  (15 features)

Todos los archivos guardados correctamente.
Siguiente paso -> 02_education_features.ipynb
